# Insurance Copilot - Google Colab Setup

This notebook sets up and runs the Insurance Copilot application on Google Colab with ngrok for public access.

## Step 1: Clone the Repository

In [ ]:
!git clone https://github.com/aksri648/AI-INSURANCE.git
%cd AI-INSURANCE

## Step 2: Install System Dependencies

In [ ]:
!apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    libpq-dev \
    tesseract-ocr \
    tesseract-ocr-eng \
    poppler-utils \
    && rm -rf /var/lib/apt/lists/*

## Step 3: Install Python Dependencies

In [ ]:
!pip install -r BACKEND/requirements.txt

## Step 4: Install and Setup Ngrok

In [ ]:
!pip install pyngrok

# Get your free ngrok auth token from https://dashboard.ngrok.com/signup
# After signing up, copy your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
from getpass import getpass
NGROK_TOKEN = getpass('Enter your ngrok authtoken: ')
!ngrok authtoken $NGROK_TOKEN

## Step 5: Setup Environment Variables

You need:
- **GROQ_API_KEY** - Get from https://console.groq.com
- **CLERK_API_KEY** - Get from https://dashboard.clerk.com
- **CLERK_JWT_PUB_KEY** - Get from https://dashboard.clerk.com (JWKS URL)
- **DATABASE_URL** - Use Neon (https://neon.tech) or any PostgreSQL with pgvector
- **TAVILY_API_KEY** (optional) - Get from https://tavily.com

In [ ]:
from getpass import getpass

GROQ_API_KEY = getpass('Enter your Groq API key: ')
CLERK_API_KEY = getpass('Enter your Clerk API key: ')
CLERK_JWT_PUB_KEY = getpass('Enter your Clerk JWKS URL: ')
DATABASE_URL = getpass('Enter your DATABASE_URL (postgresql+asyncpg://...): ')
DATABASE_URL_SYNC = getpass('Enter your DATABASE_URL_SYNC (postgresql+psycopg2://...): ')
TAVILY_API_KEY = getpass('Enter your Tavily API key (optional, press Enter to skip): ')

In [ ]:
env_content = f"""APP_NAME=Insurance Copilot
APP_VERSION=1.0.0
DEBUG=false
LOG_LEVEL=INFO
DATABASE_URL={DATABASE_URL}
DATABASE_URL_SYNC={DATABASE_URL_SYNC}
CLERK_API_KEY={CLERK_API_KEY}
CLERK_JWT_PUB_KEY={CLERK_JWT_PUB_KEY}
CLERK_WEBHOOK_SECRET=
GROQ_API_KEY={GROQ_API_KEY}
GROQ_MODEL=llama-3.1-70b-versatile
TAVILY_API_KEY={TAVILY_API_KEY}
UPLOAD_DIR=uploads
MAX_UPLOAD_SIZE_MB=50
PGVECTOR_DIMENSION=384
CHUNK_SIZE=512
CHUNK_OVERLAP=64
TOP_K_RETRIEVAL=5
CORS_ORIGINS=["*"]
RATE_LIMIT_PER_MINUTE=60
RATE_LIMIT_PER_HOUR=1000
SENTRY_DSN="""

with open('BACKEND/.env', 'w') as f:
    f.write(env_content)

print("Environment file created successfully!")

## Step 6: Build Frontend

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs

%cd /content/AI-INSURANCE/FRONTEND
!npm install
!npm run build

import shutil
import os

static_dir = '/content/AI-INSURANCE/BACKEND/static'
if os.path.exists(static_dir):
    shutil.rmtree(static_dir)
shutil.copytree('/content/AI-INSURANCE/FRONTEND/dist', static_dir)

%cd /content/AI-INSURANCE

## Step 7: Start Backend with Ngrok

In [ ]:
import subprocess
import time
from pyngrok import ngrok

ngrok.kill()

public_tunnel = ngrok.connect(8000, bind_tls=True)
public_url = public_tunnel.public_url

print(f"\n{'='*60}")
print(f"PUBLIC URL: {public_url}")
print(f"{'='*60}\n")

In [ ]:
import subprocess
import os

os.chdir('/content/AI-INSURANCE/BACKEND')

process = subprocess.Popen(
    ['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print(f"\n{'='*60}")
print(f"APPLICATION IS RUNNING")
print(f"{'='*60}")
print(f"Public URL: {public_url}")
print(f"API Docs:   {public_url}/docs")
print(f"Health:     {public_url}/health")
print(f"{'='*60}\n")

try:
    for line in process.stdout:
        print(line, end='')
except KeyboardInterrupt:
    process.terminate()
    ngrok.kill()
    print("\nServer stopped.")

## Step 8: Test Health Endpoint

In [ ]:
import requests

response = requests.get(f"{public_url}/health")
print(response.json())

## Stop Server (Run when done)

In [ ]:
process.terminate()
ngrok.kill()
print("Server and ngrok stopped.")